# Cross-cohort harmonisation: a CytoNorm-like correction for IMC (`future_plan.md` §18.7)

**Question.** Under transfer, quantile LightGBM's 80% intervals cover 0.57 (J→D) and 0.44 (D→J) (`prob-impute`). How much of that is a level/background shift that a reference-free correction of the *inputs* can remove, and how much is a change in how the markers relate?

**Setup** (as `prob-impute`: replicate-0 split and seeds, LightGBM point + 9-level quantile + residual sampling, 11 shared markers, scored on arcsinh(x/5) of the test cohort's measured values). Transfer only: Jackson → Danenberg and Danenberg → Jackson. One model per source × marker; only the mapping of the test cells changes between arms.

**Arms** (how the test cohort's inputs are brought into the training cohort's space; the target marker is never corrected):
1. **Source scaling** (= `prob-impute` transfer): test inputs scaled with the training cohort's median / (q95 − q05).
2. **Cohort z:** test inputs scaled with the test cohort's own statistics.
3. **Per-cluster ComBat** (cyCombine style): k-means on cohort-rank-normalised inputs (both cohorts pooled); within each cluster, location–scale match of each input marker, test → training cohort.
4. **Per-cluster quantile matching** (CytoNorm-style, but cohort to cohort, no reference sample): within each cluster, each input marker's test quantiles mapped to the training cohort's quantiles.
5. **GMM landmarks** (UniFORM / MxNorm style): per input marker, a 2-component Gaussian mixture per cohort; affine map that aligns the low and the high component means.
6. **Oracles (upper bounds, use the test cohort's target marginal):**
   - 6a *level + scale*: arm-2 inputs; output mapped back with the test cohort's median / scale of the target marker.
   - 6b *full marginal*: arm-4 inputs; output quantile-mapped from the training cohort's target marginal to the test cohort's.

**No leakage in arms 1–5:** every correction is fitted on the input markers only (clusters exclude the target marker); outputs go back with the training cohort's statistics of the target.

**Biology check.** Reference-free correction can remove real cohort differences. For each arm (all 15 inputs corrected): (a) cohort-classifier AUROC (training vs corrected test cells; 0.5 = cohorts indistinguishable); (b) Spearman of per-patient input means before vs after correction, within the test cohort (1 = between-patient biology kept).

**Decision (fixed before the run).** gap = mean over 10 markers of |cov80 − 0.80| for quantile LightGBM. Closed fraction = (gap₁ − gap_arm) / (gap₁ − gap_oracle), oracle = the better of 6a / 6b.
- If the best of arms 2–5 closes ≥ 50% in **both** directions, it becomes the S2 preprocessing for `bayes-impute`.
- If the oracle itself reduces the gap by < 0.03 in a direction, the failure there is not a level shift; harmonisation is dropped for it.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
from sklearn.mixture import GaussianMixture
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
REP = 0
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # cells per cohort for the point model (as train-test-grid)
N_FIT_Q = 20_000 if SMOKE else 100_000        # cells per cohort for each quantile model
N_DRAW = 50
QL = np.array([0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975])
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
LGBQ = dict(LGB, n_estimators=40 if SMOKE else 150)
NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)
assert len(MARKERS) == 11, MARKERS
EXCL = "estrogen_receptor_alpha"
SOURCES = ["Jackson", "Danenberg", "Both"]
COHORTS = ["Jackson", "Danenberg"]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

FAILED = []
def guard(name):
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
N_CLUST = 30
MIN_CL = 100        # cells per cohort in a cluster, else the global map is used

## Cell tables and Danenberg per-cell image means (as `prob-impute`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p)
P["danenberg"]["split"] = near("danenberg_patient_split.csv", P["danenberg"]["cells"])

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    return dict(cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
tick(f"tables: Danenberg {len(DD['cells']):,} cells, Jackson {len(DJ['cells']):,} cells")

if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)
import tifffile
tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    lv = loc[v]
    cnt = np.bincount(lv, minlength=len(obj))
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.stack([np.where(cnt > 0, np.bincount(lv, im[v, c], len(obj)) / np.maximum(cnt, 1), np.nan)
                         for c in range(im.shape[1])], 1).astype(np.float32)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(DD["img_names"], ST[:-1], ST[1:])))
assert RAW.shape == (len(DD["cells"]), len(D_CHAN))
dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
X = {"Danenberg": np.c_[np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF), DD["log_area"]].astype(np.float64),
     "Jackson": np.c_[np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF), DJ["log_area"]].astype(np.float64)}
del RAW
tick("per-cell image means")

## Splits and per-cohort scaling statistics (as `prob-impute`)

In [ ]:
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
dval = set(D_TRVA_PAT[uv < 0.125])
D_SPLIT = np.where(DD["split"] == "test", "test", np.where(pd.Series(DD["patient"]).isin(dval).values, "val", "train"))
SPLIT = {"Jackson": J_SPLIT, "Danenberg": D_SPLIT}
IDX = {c: {s: np.flatnonzero(SPLIT[c] == s) for s in ("train", "val", "test")} for c in COHORTS}
PAT = {"Jackson": DJ["patient"], "Danenberg": DD["patient"]}
IMG = {"Jackson": DJ["img"], "Danenberg": DD["img"]}
for c in COHORTS:
    print(c, {s: (len(v), len(set(PAT[c][v]))) for s, v in IDX[c].items()}, "(cells, patients)")

# harmonisation statistics, exactly as harm() in train-test-grid (all cells of the cohort)
STAT = {}
for c in COHORTS:
    med = np.nanmedian(X[c], 0)
    q05, q95 = np.nanpercentile(X[c], [5, 95], 0)
    STAT[c] = (med, np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X[c], 0) + 1e-6))

def H(c_data, c_stat):
    med, sc = STAT[c_stat]
    return ((X[c_data] - med) / sc).astype(np.float32)

HH = {(cd, cs): H(cd, cs) for cd in COHORTS for cs in COHORTS}
tick("splits and harmonisation")

## Scoring helpers (as `prob-impute`)

In [ ]:
def crps_draws(S, y):
    # S: (n, k) draws, y: (n,). CRPS = E|X - y| - 0.5 E|X - X'|, the unbiased ("fair") estimator over draws
    k = S.shape[1]
    Ss = np.sort(S, 1)
    t1 = np.abs(Ss - y[:, None]).mean(1)
    w = (2 * np.arange(1, k + 1) - k - 1) / (k * (k - 1))
    t2 = (Ss * w).sum(1)          # = 0.5 E|X - X'|
    return t1 - t2

def boot_mean(v, pc, n_pat, W):
    s, c = np.bincount(pc, v, n_pat), np.bincount(pc, minlength=n_pat).astype(np.float64)
    return float(s.sum() / c.sum()), (W @ s) / (W @ c)

def r4(a, b, img):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok], b[ok], img[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def interp_draws(Q, rng, n):
    # inverse-CDF draws from per-cell quantiles Q (cells × levels), linear between levels; tails held flat
    # at the outer levels beyond 0.025 / 0.975 (i.e. the outer 5% is clamped)
    uq = rng.random((Q.shape[0], n))
    out = np.empty_like(uq)
    for i in range(n):
        u_ = np.clip(uq[:, i], QL[0], QL[-1])
        j = np.clip(np.searchsorted(QL, u_) - 1, 0, len(QL) - 2)
        t = (u_ - QL[j]) / (QL[j + 1] - QL[j])
        r = np.arange(Q.shape[0])
        out[:, i] = Q[r, j] * (1 - t) + Q[r, j + 1] * t
    return out

TEST = {c: IDX[c]["test"] for c in COHORTS}
BOOTW = {}
for c in COHORTS:
    tp, tcode = np.unique(PAT[c][TEST[c]], return_inverse=True)
    W = np.random.default_rng(SEED + (12 if c == "Jackson" else 13)).multinomial(
        len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    BOOTW[c] = (tcode, len(tp), W)

## Corrections of the test cohort's inputs (arcsinh space), fitted on inputs only

In [ ]:
NQ = 101
QS = np.linspace(0, 1, NQ)

def fill(A):
    A = A.copy()
    med = np.nanmedian(A, 0)
    r, c = np.where(~np.isfinite(A))
    A[r, c] = med[c]
    return A

def rank01(A):
    return np.column_stack([(rankdata(A[:, j]) - 0.5) / len(A) for j in range(A.shape[1])])

def clusters(Xs, Xt, seed):
    # k-means on per-cohort rank-normalised inputs, so that clusters are cell states shared by both cohorts
    Rs, Rt = rank01(fill(Xs)), rank01(fill(Xt))
    rng = np.random.default_rng(seed)
    fit = np.vstack([Rs[rng.choice(len(Rs), min(100_000, len(Rs)), replace=False)],
                     Rt[rng.choice(len(Rt), min(100_000, len(Rt)), replace=False)]])
    km = MiniBatchKMeans(N_CLUST, random_state=seed, batch_size=4096, n_init=3).fit(fit)
    return km.predict(Rs), km.predict(Rt)

def quantile_match(xt, xs):
    ok_t, ok_s = np.isfinite(xt), np.isfinite(xs)
    qt, qs = np.nanquantile(xt[ok_t], QS), np.nanquantile(xs[ok_s], QS)
    qt = np.maximum.accumulate(qt + np.arange(NQ) * 1e-9)       # strictly increasing for interp
    out = xt.copy()
    out[ok_t] = np.interp(xt[ok_t], qt, qs)
    return out

def loc_scale(xt, xs):
    mt, st = np.nanmean(xt), np.nanstd(xt) + 1e-6
    ms, ss = np.nanmean(xs), np.nanstd(xs) + 1e-6
    return (xt - mt) / st * ss + ms

def per_cluster(Xt, Xs, lt, ls, fn):
    out = np.empty_like(Xt)
    for j in range(Xt.shape[1]):
        out[:, j] = fn(Xt[:, j], Xs[:, j])                      # global fallback
        for c in range(N_CLUST):
            it, is_ = np.flatnonzero(lt == c), np.flatnonzero(ls == c)
            if len(it) >= MIN_CL and len(is_) >= MIN_CL:
                out[it, j] = fn(Xt[it, j], Xs[is_, j])
    return out

def gmm_landmarks(x, seed):
    x = x[np.isfinite(x)]
    s = np.random.default_rng(seed).choice(x, min(50_000, len(x)), replace=False)[:, None]
    m = np.sort(GaussianMixture(2, random_state=seed, n_init=2).fit(s).means_.ravel())
    return m if m[1] - m[0] > 0.05 else np.nanquantile(x, [0.1, 0.9])   # unimodal → robust landmarks

def landmark_map(Xt, Xs, seed):
    out = np.empty_like(Xt)
    for j in range(Xt.shape[1]):
        lt, ls = gmm_landmarks(Xt[:, j], seed), gmm_landmarks(Xs[:, j], seed)
        out[:, j] = ls[0] + (Xt[:, j] - lt[0]) * (ls[1] - ls[0]) / max(lt[1] - lt[0], 1e-3)
    return out

def to_source_units(Xraw, c_stat):
    med, sc = STAT[c_stat]
    return ((Xraw - med) / sc).astype(np.float32)

def corrected_inputs(s, t, cols, seed):
    # returns {arm: test-cohort input matrix (all test-cohort cells × cols) in the training cohort's scaled units}
    Xs = X[s][IDX[s]["train"]][:, cols]
    Xt = X[t][:, cols]
    ls, lt = clusters(Xs, Xt, seed)
    med_s, sc_s = STAT[s][0][cols], STAT[s][1][cols]
    U = lambda A: ((A - med_s) / sc_s).astype(np.float32)
    return {"1 source scaling": HH[(t, s)][:, cols],
            "2 cohort z": HH[(t, t)][:, cols],
            "3 per-cluster ComBat": U(per_cluster(Xt, Xs, lt, ls, loc_scale)),
            "4 per-cluster quantile": U(per_cluster(Xt, Xs, lt, ls, quantile_match)),
            "5 GMM landmarks": U(landmark_map(Xt, Xs, seed))}

# unit checks on synthetic data: a pure shift + scale is undone by every map
_r = np.random.default_rng(0)
_xs = np.r_[_r.normal(0, .1, 5000), _r.normal(2, .3, 5000)]
_xt = 1.5 * _xs[_r.permutation(len(_xs))] + 0.7
assert abs(np.mean(quantile_match(_xt, _xs)) - _xs.mean()) < 0.02
assert abs(np.std(loc_scale(_xt, _xs)) - _xs.std()) < 0.02
assert abs(np.mean(landmark_map(_xt[:, None], _xs[:, None], 0)) - _xs.mean()) < 0.05
print("correction unit checks passed")

## Biology check: how much cohort signal and between-patient structure survive each correction (all 15 inputs)

In [ ]:
BIO = []
ALL = list(range(len(IN_SET) + 1))
@guard("biology check")
def _():
    for s in COHORTS:
        t = OTHER[s]
        arms = corrected_inputs(s, t, ALL, SEED + 101)
        rng = np.random.default_rng(SEED + 5)
        si = rng.choice(IDX[s]["train"], min(60_000, len(IDX[s]["train"])), replace=False)
        ti = rng.choice(len(X[t]), min(60_000, len(X[t])), replace=False)
        As = HH[(s, s)][si]
        tp, tcode = np.unique(PAT[t], return_inverse=True)
        before = HH[(t, t)]
        mb = np.column_stack([np.bincount(tcode, np.nan_to_num(before[:, j]), len(tp)) / np.bincount(tcode, minlength=len(tp))
                              for j in ALL])
        for arm, A in arms.items():
            Z = np.vstack([As, A[ti]]); lab = np.r_[np.zeros(len(si)), np.ones(len(ti))]
            perm = rng.permutation(len(Z)); half = len(Z) // 2
            tr_, te_ = perm[:half], perm[half:]
            clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, verbose=-1,
                                     n_jobs=os.cpu_count()).fit(Z[tr_], lab[tr_])
            auc = roc_auc_score(lab[te_], clf.predict_proba(Z[te_])[:, 1])
            ma = np.column_stack([np.bincount(tcode, np.nan_to_num(A[:, j]), len(tp)) / np.bincount(tcode, minlength=len(tp))
                                  for j in ALL])
            rho = [spearmanr(mb[:, j], ma[:, j]).correlation for j in ALL]
            BIO.append(dict(train=s, test=t, arm=arm, cohort_auroc=auc, patient_mean_spearman_median=float(np.nanmedian(rho)),
                            patient_mean_spearman_min=float(np.nanmin(rho)),
                            worst_input=(IN_SET + ["log_area"])[int(np.nanargmin(rho))]))
        tick(f"biology check {s} -> {t}")
    B = pd.DataFrame(BIO)
    B.to_csv(f"{OUT}/biology_check.csv", index=False)
    print(B.round(3).to_string(index=False))

## Fit once per source × marker; score every arm on the other cohort

In [ ]:
ROWS, LOG = [], []
BOOTS = {}

def score(src, test, m, arm, name, draws=None, point=None):
    k = IN_SET.index(m)
    y = X[test][TEST[test], k]
    ok = np.isfinite(y)
    tcode, n_pat, W = BOOTW[test]
    pc, y = tcode[ok], y[ok]
    img = IMG[test][TEST[test]][ok]
    rec = dict(train=src, test=test, marker=m, arm=arm, model=name)
    if draws is not None:
        S = draws[ok]
        crps = crps_draws(S, y)
        for lev in (0.5, 0.8, 0.95):
            lo, hi = np.quantile(S, (1 - lev) / 2, 1), np.quantile(S, 1 - (1 - lev) / 2, 1)
            cov = ((y >= lo) & (y <= hi)).astype(np.float64)
            pt, bv = boot_mean(cov, pc, n_pat, W)
            rec[f"cov{int(lev * 100)}"] = pt
            BOOTS[(src, test, m, arm, name, f"cov{int(lev * 100)}")] = (pt, bv)
            if lev == 0.8:
                rec["width80"] = float(np.mean(hi - lo))
        Ss = np.sort(S, 1)
        pit = ((Ss < y[:, None]).sum(1) + np.random.default_rng(SEED + 7).random(len(y)) * ((Ss == y[:, None]).sum(1) + 1)) / (S.shape[1] + 1)
        ps = np.sort(pit)
        rec["pit_ks"] = float(np.max(np.abs(ps - (np.arange(1, len(ps) + 1) - 0.5) / len(ps))))
        one = S[np.arange(len(y)), np.random.default_rng(SEED + 9).integers(0, S.shape[1], len(y))]
        rec["W2_draw"] = float(np.sqrt(np.mean((np.sort(one) - np.sort(y)) ** 2)))
        rec["R4"] = r4(y, np.median(S, 1), img)
    else:
        p = point[ok]
        crps = np.abs(p - y)
        se = (p - y) ** 2
        pt, bv = boot_mean(se, pc, n_pat, W)
        rec["rmse"] = float(np.sqrt(pt))
        BOOTS[(src, test, m, arm, name, "mse")] = (pt, bv)
        rec["W2_draw"] = float(np.sqrt(np.mean((np.sort(p) - np.sort(y)) ** 2)))
        if np.std(p) > 0:
            rec["R4"] = r4(y, p, img)
    pt, bv = boot_mean(crps, pc, n_pat, W)
    rec["crps"] = pt
    BOOTS[(src, test, m, arm, name, "crps")] = (pt, bv)
    ROWS.append(rec)

for s in COHORTS:
    t = OTHER[s]
    tick(f"train on {s}, test on {t}")
    for m in MARKERS:
        k = IN_SET.index(m)
        cols = [i for i in range(len(IN_SET) + 1) if i != k]
        Hs = HH[(s, s)]
        rs = np.random.default_rng(SEED + REP)
        tr = IDX[s]["train"][np.isfinite(Hs[IDX[s]["train"], k])]
        f1 = np.sort(rs.choice(tr, min(N_FIT_FINAL, len(tr)), replace=False))
        fq = np.sort(rs.choice(tr, min(N_FIT_Q, len(tr)), replace=False))
        t0 = time.time()
        point = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Hs[f1][:, cols], Hs[f1, k])
        qms = [lgb.LGBMRegressor(objective="quantile", alpha=float(a), random_state=SEED + REP, **LGBQ).fit(Hs[fq][:, cols], Hs[fq, k])
               for a in QL]
        va = IDX[s]["val"][np.isfinite(Hs[IDX[s]["val"], k])]
        rp = point.predict(Hs[va][:, cols]); rr = Hs[va, k] - rp
        edges = np.quantile(rp, np.linspace(0, 1, 11)[1:-1])
        dec = np.searchsorted(edges, rp)
        pools = [rr[dec == d] for d in range(10)]
        clim = Hs[f1, k]
        arms = corrected_inputs(s, t, cols, SEED + 101 + k)
        arms["6a oracle: level + scale"] = arms["2 cohort z"]
        arms["6b oracle: full marginal"] = arms["4 per-cluster quantile"]
        # back-maps of the target marker
        back_s = lambda v: v * STAT[s][1][k] + STAT[s][0][k]
        back_t = lambda v: v * STAT[t][1][k] + STAT[t][0][k]
        ys_marg = X[s][IDX[s]["train"], k]; yt_marg = X[t][:, k]
        qs_, qt_ = np.nanquantile(ys_marg, np.linspace(0, 1, 1001)), np.nanquantile(yt_marg, np.linspace(0, 1, 1001))
        qs_ = np.maximum.accumulate(qs_ + np.arange(1001) * 1e-9)
        back_m = lambda v: np.interp(back_s(v), qs_, qt_)
        BACK = {a: (back_t if a.startswith("6a") else back_m if a.startswith("6b") else back_s) for a in arms}
        for arm, A in arms.items():
            Hin = A[TEST[t]]
            back = BACK[arm]
            rng = np.random.default_rng(SEED + 31)
            pp = point.predict(Hin)
            score(s, t, m, arm, "Point LightGBM", point=back(pp))
            Q = np.sort(np.stack([q.predict(Hin) for q in qms], 1), 1)
            score(s, t, m, arm, "Quantile LightGBM", draws=back(interp_draws(Q, rng, N_DRAW)))
            d = np.searchsorted(edges, pp)
            Sres = np.empty((len(pp), N_DRAW))
            for dd in range(10):
                ii = np.flatnonzero(d == dd)
                if len(ii):
                    Sres[ii] = pp[ii, None] + rng.choice(pools[dd], (len(ii), N_DRAW))
            score(s, t, m, arm, "Residual sampling", draws=back(Sres))
            if arm.startswith(("1", "6")):          # baselines depend only on the output map
                score(s, t, m, arm, "Constant (training mean)", point=np.full(len(pp), back(clim.mean())))
                score(s, t, m, arm, "Climatology", draws=back(rng.choice(clim, (len(pp), N_DRAW))))
        LOG.append(dict(train=s, marker=m, seconds=time.time() - t0))
        tick(f"  {s}->{t}: {m}")
    pd.DataFrame(ROWS).to_csv(f"{OUT}/scores_per_marker.csv", index=False)
pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)

## Summaries, paired differences vs arm 1, and the decision

In [ ]:
R = pd.DataFrame(ROWS)
R10 = R[R.marker != EXCL]
cols_ = ["crps", "cov50", "cov80", "cov95", "width80", "pit_ks", "W2_draw", "R4", "rmse"]
SUM = R10.groupby(["test", "train", "arm", "model"])[[c for c in cols_ if c in R10]].mean().reset_index()
SUM["gap80"] = R10.assign(g=(R10.cov80 - 0.8).abs()).groupby(["test", "train", "arm", "model"]).g.mean().values
SUM.to_csv(f"{OUT}/summary_mean10.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(SUM.round(3).to_string(index=False))

ms10 = [m for m in MARKERS if m != EXCL]
ARMS = sorted(R.arm.unique())
AGG, DIFF = [], []
for s in COHORTS:
    t = OTHER[s]
    for arm in ARMS:
        for name in R.model.unique():
            for met in ["crps", "cov80", "mse"]:
                keys = [(s, t, m, arm, name, met) for m in ms10]
                if not all(k_ in BOOTS for k_ in keys):
                    continue
                bv = np.mean([BOOTS[k_][1] for k_ in keys], 0)
                AGG.append(dict(train=s, test=t, arm=arm, model=name, metric=met,
                                value=np.mean([BOOTS[k_][0] for k_ in keys]), lo=np.percentile(bv, 2.5), hi=np.percentile(bv, 97.5)))
                if met == "cov80":   # gap = mean |cov80 - 0.8|, with CI
                    gv = np.mean([np.abs(BOOTS[k_][1] - 0.8) for k_ in keys], 0)
                    AGG.append(dict(train=s, test=t, arm=arm, model=name, metric="gap80",
                                    value=np.mean([abs(BOOTS[k_][0] - 0.8) for k_ in keys]),
                                    lo=np.percentile(gv, 2.5), hi=np.percentile(gv, 97.5)))
        if arm.startswith("1"):
            continue
        for name in ["Point LightGBM", "Quantile LightGBM", "Residual sampling"]:
            for m in ms10:
                a, b = BOOTS[(s, t, m, arm, name, "crps")], BOOTS[(s, t, m, "1 source scaling", name, "crps")]
                lo, hi = np.percentile(a[1] - b[1], [2.5, 97.5])
                DIFF.append(dict(train=s, test=t, arm=arm, model=name, marker=m, diff=a[0] - b[0], lo=lo, hi=hi,
                                 call="better" if hi < 0 else "worse" if lo > 0 else "n.s."))
AGG, DIFF = pd.DataFrame(AGG), pd.DataFrame(DIFF)
AGG.to_csv(f"{OUT}/summary_mean10_ci.csv", index=False)
DIFF.to_csv(f"{OUT}/crps_vs_arm1.csv", index=False)
print(AGG[AGG.model == "Quantile LightGBM"].pivot_table(index=["test", "arm"], columns="metric", values="value").round(3).to_string())
print(DIFF.groupby(["test", "arm", "model"]).call.value_counts().unstack(fill_value=0).to_string())

# decision (fixed before the run)
q = AGG[(AGG.model == "Quantile LightGBM") & (AGG.metric == "gap80")].set_index(["test", "arm"]).value
dec = {}
for t in COHORTS:
    g1 = q[(t, "1 source scaling")]
    go = min(q[(t, "6a oracle: level + scale")], q[(t, "6b oracle: full marginal")])
    cand = {a: q[(t, a)] for a in ARMS if a[0] in "2345"}
    best = min(cand, key=cand.get)
    dec[t] = dict(gap_arm1=g1, gap_oracle=go, oracle_reduction=g1 - go, best_arm=best, gap_best=cand[best],
                  closed_fraction=(g1 - cand[best]) / (g1 - go) if g1 - go > 1e-9 else float("nan"),
                  level_shift_explains=bool(g1 - go >= 0.03))
adopt = all(d["level_shift_explains"] and d["closed_fraction"] >= 0.5 for d in dec.values())
best_both = {d["best_arm"] for d in dec.values()}
decision = dict(per_test_cohort=dec, adopt_for_bayes_impute=adopt,
                arm=(best_both.pop() if adopt and len(best_both) == 1 else None),
                rule="best of arms 2-5 closes >= 50% of the arm-1 -> oracle gap in |cov80-0.8| in both directions")
print(json.dumps(decision, indent=1, default=float))
json.dump(decision, open(f"{OUT}/decision.json", "w"), indent=1, default=float)

## Figure: 80% coverage gap and CRPS by arm

In [ ]:
@guard("figure")
def _():
    names = ["Point LightGBM", "Quantile LightGBM", "Residual sampling"]
    colr = {"Point LightGBM": "#52514e", "Quantile LightGBM": "#2a78d6", "Residual sampling": "#008300"}
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for i, s in enumerate(COHORTS):
        t = OTHER[s]
        for j, met in enumerate(["gap80", "crps"]):
            ax = axes[j, i]
            nm_list = names[1:] if met == "gap80" else names
            for qn, nm in enumerate(nm_list):
                xs, vs, los, his = [], [], [], []
                for p_, arm in enumerate(ARMS):
                    r = AGG[(AGG.train == s) & (AGG.arm == arm) & (AGG.model == nm) & (AGG.metric == met)]
                    if len(r):
                        xs.append(p_ + (qn - (len(nm_list) - 1) / 2) * 0.25)
                        vs.append(r.value.iloc[0]); los.append(r.lo.iloc[0]); his.append(r.hi.iloc[0])
                vs, los, his = map(np.array, (vs, los, his))
                ax.bar(xs, vs, width=0.22, color=colr[nm], label=nm)
                ax.errorbar(xs, vs, yerr=[vs - los, his - vs], fmt="none", ecolor="#0b0b0b", elinewidth=0.8, capsize=2)
            ax.axvline(len(ARMS) - 2.5, color="#b0aea7", lw=1, ls=(0, (4, 3)))
            ax.set_xticks(range(len(ARMS)))
            ax.set_xticklabels([a.replace("oracle: ", "oracle:\n") for a in ARMS], rotation=30, ha="right", fontsize=8)
            ax.set_ylabel(("Mean |80% coverage − 0.80|" if met == "gap80" else "CRPS, arcsinh units") +
                          f"\n{s} → {t} (lower is better)")
            ax.spines[["top", "right"]].set_visible(False)
    h, l = axes[1, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(f"{OUT}/fig_harmonise.png", dpi=200)

json.dump({"failed_blocks": FAILED, "minutes": (time.time() - T0) / 60}, open(f"{OUT}/run_info.json", "w"))
print("failed blocks:", FAILED if FAILED else "none")